<div class="jupyter-biolm-header">
    <img style="float: left; padding-right: 10px; height: 60px" src="https://d31e6ufxekikrt.cloudfront.net/static/ui/images/logo.png">
    <p>
    <br>
    <br>
    <br>
    </p>
</div>

# Why DuckDB Inside the SDK

How the SDK uses DuckDB for caching, deduplication, and zero-cost re-querying.

<br>

<table class="jupyter-biolm-header-table" style="width: 100%; border-collapse: collapse; background-color: white; float: left;">
    <tr>
        <td style="text-align: left; vertical-align: middle; background-color: white;">
            <img src="https://www.svgrepo.com/show/354202/postman-icon.svg" style="height: 15px; float: left; padding-right: 10px;"><a href="https://api.biolm.ai/">  <h5 style="margin: 0;"><b>Postman API Docs</b></h5></a>
        </td>
        <td style="text-align: left; vertical-align: middle; background-color: white;">
            <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/c/c3/Python-logo-notext.svg/1869px-Python-logo-notext.svg.png" style="height: 15px; float: left; padding-right: 10px;"><a href="https://docs.biolm.ai/en/latest/index.html"><h5 style="margin: 0;"><b>Python SDK Docs</b></h5></a>
        </td>
        <td style="text-align: left; vertical-align: middle; background-color: white;">
        </td>
        <td style="text-align: left; vertical-align: middle; background-color: white;">
        </td>
    </tr>
</table>

<br>

---

> **⚠️ Preview Feature** — The `biolmai.pipeline` module used in this guide is currently in preview and not yet publicly released. Access is available to early users on request. [Contact us](https://biolm.ai) to get access.

**What you'll learn:**
- How sequence deduplication and cache lookup work under the hood
- How to query the DuckDB store directly with arbitrary SQL
- How to resume a pipeline after a crash or session restart

**Requirements:**
```
pip install biolmai[pipeline]
export BIOLMAI_TOKEN=your-token-here
```

## Setup

In [ ]:
import os
from biolmai.pipeline import (
    DataPipeline, DuckDBDataStore,
    ThresholdFilter, RankingFilter,
    ValidAminoAcidFilter, EmbeddingSpec,
    DiversitySamplingFilter,
)

TOKEN = os.environ.get("BIOLMAI_TOKEN", "")
if not TOKEN:
    raise EnvironmentError(
        "Set BIOLMAI_TOKEN before running.\n"
        "Get one at https://biolm.ai/ui/accounts/user-api-tokens/"
    )

## The cache in action

Run a pipeline, then re-run it. The second run fires zero API calls.

In [ ]:
PEPTIDES = [
    "GIGKFLHSAKKFGKAFVGEIMNS",
    "GLFDIIKKIAESF",
    "KLAKLAKKLAKLAK",
    "RRWWRRWWRR",
    "KWKLFKKI",
]

ds = DuckDBDataStore("sdk_demo.duckdb")

pipeline = DataPipeline(sequences=PEPTIDES, datastore=ds, run_id="run_v1", verbose=True)
pipeline.add_prediction("temperature-regression", extractions="prediction", columns="melting_temperature")
pipeline.add_prediction("biolmsol", extractions="solubility_score", columns="solubility")
pipeline.run()
print("Run 1 complete — check the API call count above")

In [ ]:
# Re-run identical pipeline — 0 API calls
pipeline2 = DataPipeline(sequences=PEPTIDES, datastore=ds, run_id="run_v1", verbose=True)
pipeline2.add_prediction("temperature-regression", extractions="prediction", columns="melting_temperature")
pipeline2.add_prediction("biolmsol", extractions="solubility_score", columns="solubility")
pipeline2.run()
print("Run 2 complete — all served from cache")

## Query the DuckDB store directly

The `.duckdb` file is a standard DuckDB database. You can run arbitrary SQL against it.

In [ ]:
# Schema overview
ds.conn.execute("""
    SELECT prediction_type, model_name, COUNT(*) AS n, 
           ROUND(AVG(value), 3) AS mean, ROUND(MIN(value), 3) AS min, ROUND(MAX(value), 3) AS max
    FROM predictions
    GROUP BY prediction_type, model_name
""").df()

In [ ]:
# Full prediction history for every sequence
ds.conn.execute("""
    SELECT s.sequence, p.prediction_type, ROUND(p.value, 3) AS value
    FROM sequences s
    JOIN predictions p ON s.sequence_id = p.sequence_id
    ORDER BY s.sequence, p.prediction_type
""").df()

## Re-filter without re-predicting

Change your filter threshold and re-run — predictions are already cached.

In [ ]:
pipeline3 = DataPipeline(sequences=PEPTIDES, datastore=ds, run_id="run_v3_strict", verbose=True)
pipeline3.add_prediction("temperature-regression", extractions="prediction", columns="melting_temperature")
pipeline3.add_prediction("biolmsol", extractions="solubility_score", columns="solubility")
pipeline3.add_filter(ThresholdFilter("melting_temperature", min_value=50.0))  # stricter threshold
pipeline3.run()
pipeline3.summary()

## Cleanup

In [ ]:
ds.close()
import os; os.remove("sdk_demo.duckdb")

## Next Steps

Check out additional tutorials at [jupyter.biolm.ai](https://jupyter.biolm.ai),
or head over to our [BioLM Documentation](https://docs.biolm.ai) to explore
additional models and functionality.

#### See more use-cases and APIs on your [BioLM Console Catalog](https://biolm.ai/console/catalog/).
<br>

##### BioLM hosts deep learning models and runs inference at scale. You do the science.
<br>

<table class="jupyter-biolm-header-table" style="width: 100%; border-collapse: collapse; background-color: white; float: left;">
    <tr>
        <td style="text-align: left; vertical-align: middle; background-color: white;">
            <img src="https://d31e6ufxekikrt.cloudfront.net/static/ui/images/console-overview/enzyme_engineering_icon.png"  style="height: 40px; float: left; padding-right: 10px;"> Enzyme Engineering
        </td>
        <td style="text-align: left; vertical-align: middle; background-color: white;">
            <img src="https://d31e6ufxekikrt.cloudfront.net/static/ui/images/console-overview/antibody_engineering_icon.png"  style="height: 40px; float: left; padding-right: 10px;"> Antibody Engineering
        </td>
        <td style="text-align: left; vertical-align: middle; background-color: white;">
            <img src="https://d31e6ufxekikrt.cloudfront.net/static/ui/images/console-overview/biosecurity_icon.png"  style="height: 40px; float: left; padding-right: 10px;"> Biosecurity
        </td>
    </tr>
    <tr>
        <td style="text-align: left; vertical-align: middle; background-color: white;">
            <img src="https://d31e6ufxekikrt.cloudfront.net/static/ui/images/console-overview/single_cell_genomics_icon.png"  style="height: 40px; float: left; padding-right: 10px;"> Single-Cell Genomics
        </td>
        <td style="text-align: left; vertical-align: middle; background-color: white;">
            <img src="https://d31e6ufxekikrt.cloudfront.net/static/ui/images/console-overview/dna_seq_modeling_icon.png"  style="height: 40px; float: left; padding-right: 10px;"> DNA Sequence Modelling
        </td>
        <td style="text-align: left; vertical-align: middle; background-color: white;">
            <img src="https://d31e6ufxekikrt.cloudfront.net/static/ui/images/console-overview/finetuning_icon.png"  style="height: 40px; float: left; padding-right: 10px;"> Finetuning
        </td>
    </tr>
</table>

#### [**Contact us**](https://biolm.ai/ui/contact-us/) to learn more.